In [ ]:

!pip install -q --upgrade kaggle
!pip install -q torch torchaudio audiomentations scikit-learn pandas numpy tqdm wandb
!pip install -q soundfile

In [2]:
import os
os.environ["WANDB_API_KEY"] = "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"

In [3]:
# ── CLEANUP ONLY — wipe datasets, keys, manifests, checkpoints ──
import os, shutil

targets = [
    '/content/data',                 # 2021 DF FLACs
    '/content/keys',                 # extracted DF keys
    '/content/DF-keys-full.tar.gz',  # keys tarball
    '/content/manifest_df2021.csv',  # old manifest
    '/content/manifest_df2021_fixed.csv',
    '/content/checkpoints',          # model checkpoints
    '/content/wandb',                # local wandb run cache
]

for p in targets:
    if os.path.isdir(p):
        shutil.rmtree(p, ignore_errors=True); print('rm -rf', p)
    elif os.path.isfile(p):
        try: os.remove(p); print('rm -f ', p)
        except FileNotFoundError: pass
    else:
        print('skip  ', p, '(not present)')

print('\nClean. Nothing left under /content/data, /content/keys, or manifests.')

rm -rf /content/data
rm -rf /content/keys
rm -f  /content/DF-keys-full.tar.gz
rm -f  /content/manifest_df2021.csv
skip   /content/manifest_df2021_fixed.csv (not present)
rm -rf /content/checkpoints
skip   /content/wandb (not present)

Clean. Nothing left under /content/data, /content/keys, or manifests.


In [4]:
import os
from getpass import getpass

token = getpass('Paste FULL Kaggle API token (KGAT_...): ').strip()
os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
tok_path = os.path.expanduser('~/.kaggle/access_token')
with open(tok_path, 'w') as f:
    f.write(token)
os.chmod(tok_path, 0o600)
os.environ['KAGGLE_API_TOKEN'] = token

# Verify
!kaggle datasets list -m | head

Paste FULL Kaggle API token (KGAT_...):  ········


ref                            title                       size  lastUpdated                 downloadCount  voteCount  usabilityRating  
-----------------------------  --------------------  ----------  --------------------------  -------------  ---------  ---------------  
jeetsubh/asvspoof2021-df-eval  ASVspoof2021 DF Eval  8644430723  2026-09-15 21:27:30.783000              7          0  0.25             


In [5]:
import os, glob, subprocess

DATA_DIR = '/content/data'
os.makedirs(DATA_DIR, exist_ok=True)

subprocess.run([
    'kaggle','datasets','download',
    '-d','jeetsubh/asvspoof2021-df-eval',
    '-p', DATA_DIR, '--unzip'
], check=True)

# Extract any nested zips
for z in glob.glob(f'{DATA_DIR}/**/*.zip', recursive=True):
    print('Unzipping:', z)
    subprocess.run(['unzip','-q','-o',z,'-d',os.path.dirname(z)], check=True)

n_flac = len(glob.glob(f'{DATA_DIR}/**/*.flac', recursive=True))
print('FLAC count:', n_flac)   # expect 152955
print('Top-level :', os.listdir(DATA_DIR))

Dataset URL: https://www.kaggle.com/datasets/jeetsubh/asvspoof2021-df-eval
License(s): CC0-1.0


100%|██████████| 8.05G/8.05G [00:52<00:00, 166MB/s] 



FLAC count: 152955
Top-level : ['README.txt', 'ASVspoof2021.DF.cm.eval.trl.txt', 'README.DF.txt', 'flac', 'LICENSE.DF.txt']


In [6]:
import subprocess, glob, os

KEY_URL = 'https://www.asvspoof.org/asvspoof2021/DF-keys-full.tar.gz'
KEY_TAR = '/content/DF-keys-full.tar.gz'

# Try wget, fall back to curl
r = subprocess.run(['wget','-q',KEY_URL,'-O',KEY_TAR])
if r.returncode != 0 or not os.path.exists(KEY_TAR) or os.path.getsize(KEY_TAR) < 1000:
    print('wget failed, trying curl...')
    subprocess.run(['curl','-L','-o',KEY_TAR,KEY_URL], check=True)

print(subprocess.run(['ls','-la',KEY_TAR], capture_output=True, text=True).stdout)

# Verify checksum (official MD5)
print(subprocess.run(['md5sum',KEY_TAR], capture_output=True, text=True).stdout)
# expect: dabbc5628de4fcef53036c99ac7ab93a

# Extract
subprocess.run(['tar','-C','/content','-xf',KEY_TAR], check=True)

# Find the metadata file
cands = glob.glob('/content/**/trial_metadata.txt', recursive=True) \
      + glob.glob('/content/keys/**/*.txt', recursive=True) \
      + glob.glob('/content/**/*DF*key*.txt', recursive=True)
print('Candidates:', cands)

if not cands:
    raise RuntimeError('No key file found after extraction.')

KEY_FILE = cands[0]
print('Using KEY_FILE =', KEY_FILE)
print('--- first 5 lines ---')
with open(KEY_FILE) as f:
    for _ in range(5):
        print(repr(f.readline()))

-rw-r--r-- 1 root root 25985925 Nov 28  2024 /content/DF-keys-full.tar.gz

dabbc5628de4fcef53036c99ac7ab93a  /content/DF-keys-full.tar.gz

Candidates: ['/content/keys/DF/CM/trial_metadata.txt', '/content/keys/DF/README.txt', '/content/keys/DF/CM/trial_metadata.txt', '/content/keys/DF/CM/LFCC-LCNN/score.txt', '/content/keys/DF/CM/LFCC-GMM/score.txt', '/content/keys/DF/CM/RawNet2/score.txt', '/content/keys/DF/CM/CQCC-GMM/score.txt']
Using KEY_FILE = /content/keys/DF/CM/trial_metadata.txt
--- first 5 lines ---
'LA_0023 DF_E_2000011 nocodec asvspoof A14 spoof notrim progress traditional_vocoder - - - -\n'
'TEF2 DF_E_2000013 low_m4a vcc2020 Task1-team20 spoof notrim eval neural_vocoder_nonautoregressive Task1 team20 FF E\n'
'TGF1 DF_E_2000024 mp3m4a vcc2020 Task2-team12 spoof notrim eval traditional_vocoder Task2 team12 FF G\n'
'LA_0043 DF_E_2000026 mp3m4a asvspoof A09 spoof notrim eval traditional_vocoder - - - -\n'
'LA_0021 DF_E_2000027 mp3m4a asvspoof A12 spoof notrim eval neural_vocoder

In [7]:
import csv, os, glob
from collections import Counter

MANIFEST = '/content/manifest_df2021.csv'
FLAC_DIR = f'{DATA_DIR}/flac'

rows = []
n_no_flac = 0
with open(KEY_FILE) as f:
    for line in f:
        parts = line.strip().split()
        if len(parts) < 6:
            continue
        # DF trial_metadata.txt field layout:
        # 0 speaker | 1 trial_id | 2 codec | 3 source | 4 attack_id
        # 5 label    | 6 trim    | 7 subset | 8 vocoder | ...
        speaker_id = parts[0]
        trial_id   = parts[1]
        attack_id  = parts[4]
        key        = parts[5].lower()
        subset     = parts[7] if len(parts) > 7 else 'eval'

        path = os.path.join(FLAC_DIR, f'{trial_id}.flac')
        if not os.path.exists(path):
            n_no_flac += 1
            continue

        rows.append({
            'path':         path,
            'label':        1 if key == 'bonafide' else 0,
            'speaker_id':   speaker_id,
            'generator_id': attack_id,
            'split':        'unassigned',
        })

with open(MANIFEST, 'w', newline='') as f:
    w = csv.DictWriter(f, fieldnames=['path','label','speaker_id','generator_id','split'])
    w.writeheader(); w.writerows(rows)

print('Manifest rows :', len(rows))
print('Metadata rows without matching FLAC:', n_no_flac)
print('Labels        :', Counter(r['label'] for r in rows))
print('Unique spk    :', len({r['speaker_id'] for r in rows}))
print('Unique attack :', len({r['generator_id'] for r in rows}))

Manifest rows : 152955
Metadata rows without matching FLAC: 458874
Labels        : Counter({0: 147420, 1: 5535})
Unique spk    : 93
Unique attack : 111


In [8]:
import random, csv
from collections import Counter
random.seed(0)

speakers = sorted({r['speaker_id'] for r in rows if r['speaker_id'] != 'unknown'})
print('Speakers for grouping:', len(speakers))

if len(speakers) >= 10:
    random.shuffle(speakers)
    n = len(speakers)
    tr_end, dv_end = int(0.70*n), int(0.85*n)
    train_spk = set(speakers[:tr_end])
    dev_spk   = set(speakers[tr_end:dv_end])
    test_spk  = set(speakers[dv_end:])
else:
    # Fallback: random file split (document this as leaky)
    print('⚠️  Too few speakers for grouped split — using random file split (LEAKY).')
    idx = list(range(len(rows))); random.shuffle(idx)
    n = len(idx)
    tr_end, dv_end = int(0.70*n), int(0.85*n)
    for i, r in enumerate(rows):
        r['split'] = 'train' if i in set(idx[:tr_end]) else \
                     'dev'   if i in set(idx[tr_end:dv_end]) else 'test'
    train_spk = dev_spk = test_spk = set()

if train_spk:
    for r in rows:
        s = r['speaker_id']
        r['split'] = ('train' if s in train_spk else
                      'dev'   if s in dev_spk   else
                      'test'  if s in test_spk  else 'drop')
    assert train_spk.isdisjoint(dev_spk)
    assert train_spk.isdisjoint(test_spk)
    assert dev_spk.isdisjoint(test_spk)

with open(MANIFEST,'w',newline='') as f:
    w = csv.DictWriter(f, fieldnames=['path','label','speaker_id','generator_id','split'])
    w.writeheader(); w.writerows(rows)

print('Split counts:', Counter(r['split'] for r in rows))

Speakers for grouping: 93
Split counts: Counter({'train': 113620, 'dev': 26029, 'test': 13306})


In [9]:
import torch, torchaudio, pandas as pd
from torch.utils.data import Dataset, DataLoader
import numpy as np

SR      = 16000
DUR     = 4.0
N_MELS  = 128
N_FFT   = 1024
HOP     = 256

mel = torchaudio.transforms.MelSpectrogram(
    sample_rate=SR, n_fft=N_FFT, hop_length=HOP, n_mels=N_MELS, power=2.0)
db  = torchaudio.transforms.AmplitudeToDB(top_db=80)

class DF2021(Dataset):
    def __init__(self, df, train=False):
        self.df = df.reset_index(drop=True)
        self.train = train
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        wav, sr = torchaudio.load(r['path'])
        if wav.shape[0] > 1: wav = wav.mean(0, keepdim=True)
        if sr != SR: wav = torchaudio.functional.resample(wav, sr, SR)
        target = int(SR*DUR)
        if wav.shape[1] < target:
            wav = torch.nn.functional.pad(wav, (0, target-wav.shape[1]))
        else:
            wav = wav[:, :target]
        m = db(mel(wav)).squeeze(0)   # (N_MELS, T)
        return m, int(r['label'])

df_all = pd.read_csv(MANIFEST)
for split in ['train','dev','test']:
    print(split, (df_all['split']==split).sum())

train_ds = DF2021(df_all[df_all['split']=='train'])
dev_ds   = DF2021(df_all[df_all['split']=='dev'])
test_ds  = DF2021(df_all[df_all['split']=='test'])

train_dl = DataLoader(train_ds, batch_size=32, shuffle=True,  num_workers=2, pin_memory=True)
dev_dl   = DataLoader(dev_ds,   batch_size=64, shuffle=False, num_workers=2, pin_memory=True)
test_dl  = DataLoader(test_ds,  batch_size=64, shuffle=False, num_workers=2, pin_memory=True)

xb, yb = next(iter(train_dl))
print('Batch shape:', xb.shape, 'labels:', yb[:8])

train 113620
dev 26029
test 13306
Batch shape: torch.Size([32, 128, 251]) labels: tensor([0, 0, 0, 0, 0, 0, 0, 0])


In [10]:
import torch.nn as nn

class MelCNN(nn.Module):
    def __init__(self, n_mels=128):
        super().__init__()
        def block(i,o):
            return nn.Sequential(
                nn.Conv2d(i,o,3,padding=1), nn.BatchNorm2d(o), nn.ReLU(),
                nn.MaxPool2d(2))
        self.net = nn.Sequential(
            block(1,32), block(32,64), block(64,128), block(128,256),
            nn.AdaptiveAvgPool2d((1,1)))
        self.fc = nn.Linear(256, 2)
    def forward(self, x):
        x = x.unsqueeze(1)           # (B,1,M,T)
        x = self.net(x).flatten(1)
        return self.fc(x)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = MelCNN().to(device)
print('Device:', device, '| params:', sum(p.numel() for p in model.parameters()))

Device: cuda | params: 389314


In [11]:
import os
print('CWD        :', os.getcwd())
print('/content   :', os.path.exists('/content'))
print('/kaggle/working:', os.listdir('/kaggle/working')[:5])
print('DATA_DIR   :', DATA_DIR if 'DATA_DIR' in dir() else 'undefined')
print('FLAC count :', len(__import__('glob').glob('/content/data/flac/*.flac')) if os.path.exists('/content/data/flac') else 'no /content/data')

CWD        : /kaggle/working
/content   : True
/kaggle/working: ['wandb', '.virtual_documents']
DATA_DIR   : /content/data
FLAC count : 152955


In [12]:
import glob, os
from collections import Counter

with open(KEY_FILE) as f:
    lines = [l for l in f if l.strip()]

print('Metadata total lines:', len(lines))
print('Label distribution  :', Counter(l.split()[5].lower() for l in lines if len(l.split())>=6))
print('Subset distribution :', Counter(l.split()[7] for l in lines if len(l.split())>=8))
print('FLAC files          :', len(glob.glob(f'{FLAC_DIR}/*.flac')))

Metadata total lines: 611829
Label distribution  : Counter({'spoof': 589212, 'bonafide': 22617})
Subset distribution : Counter({'eval': 533928, 'progress': 59325, 'hidden': 18576})
FLAC files          : 152955


In [13]:
import wandb, time, os
from sklearn.metrics import roc_auc_score
import numpy as np

CKPT_DIR = '/content/checkpoints'
os.makedirs(CKPT_DIR, exist_ok=True)

# wandb.login()  # uncomment if not already logged in
RUN_NAME = 'smoke_df2021_melcnn'
wandb.init(project='streamguard', name=RUN_NAME, config={
    'sr':SR,'dur':DUR,'n_mels':N_MELS,'n_fft':N_FFT,'hop':HOP,
    'batch':32,'epochs':10,'lr':1e-3})

EPOCHS = 10
opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
scaler = torch.cuda.amp.GradScaler(enabled=(device=='cuda'))
crit = nn.CrossEntropyLoss()

def evaluate(dl):
    model.eval(); losses=[]; logits=[]; ys=[]
    with torch.no_grad():
        for x,y in dl:
            x,y = x.to(device), y.to(device)
            with torch.cuda.amp.autocast(enabled=(device=='cuda')):
                o = model(x); l = crit(o,y)
            losses.append(l.item()*y.size(0))
            logits.append(o.softmax(-1)[:,1].cpu().numpy())
            ys.append(y.cpu().numpy())
    logits = np.concatenate(logits); ys = np.concatenate(ys)
    return sum(losses)/len(ys), roc_auc_score(ys, logits), ys, logits

best_auc = 0.0
for ep in range(EPOCHS):
    model.train(); t0=time.time(); tot=0; n=0
    for x,y in train_dl:
        x,y = x.to(device), y.to(device)
        opt.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=(device=='cuda')):
            o = model(x); loss = crit(o,y)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        tot += loss.item()*y.size(0); n += y.size(0)
    sched.step()
    tr_loss = tot/n
    dv_loss, dv_auc, _, _ = evaluate(dev_dl)
    print(f'ep{ep:02d} tr_loss={tr_loss:.4f} dv_loss={dv_loss:.4f} dv_auc={dv_auc:.4f} ({time.time()-t0:.1f}s)')
    wandb.log({'ep':ep,'train_loss':tr_loss,'dev_loss':dv_loss,'dev_auc':dv_auc})
    torch.save({'model':model.state_dict(),'ep':ep,'auc':dv_auc},
               f'{CKPT_DIR}/last.pt')
    if dv_auc > best_auc:
        best_auc = dv_auc
        torch.save({'model':model.state_dict(),'ep':ep,'auc':dv_auc},
                   f'{CKPT_DIR}/best.pt')

print('Best dev AUC:', best_auc)

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: subhrajeet015 (subhrajeet015-student) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


/tmp/ipykernel_216866/2454394590.py:17: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(device=='cuda'))
/tmp/ipykernel_216866/2454394590.py:39: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device=='cuda')):


RuntimeError: Caught RuntimeError in DataLoader worker process 0.
Original Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torchaudio/_torchcodec.py", line 128, in load_with_torchcodec
    audio_samples = decoder.get_all_samples()
                    ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torchcodec/decoders/_audio_decoder.py", line 108, in get_all_samples
    return self.get_samples_played_in_range()
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torchcodec/decoders/_audio_decoder.py", line 137, in get_samples_played_in_range
    frames, first_pts = core.get_frames_by_pts_in_range_audio(
                        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/_ops.py", line 819, in __call__
    return self._op(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
RuntimeError: Could not receive frame from decoder: Invalid data

In [ ]:
from scipy.optimize import brentq
from scipy.interpolate import interp1d
from sklearn.metrics import roc_curve, auc as sk_auc, precision_recall_curve, f1_score

ckpt = torch.load(f'{CKPT_DIR}/best.pt', map_location=device)
model.load_state_dict(ckpt['model'])

_, test_auc, ys, scores = evaluate(test_dl)

# EER
fpr, tpr, _ = roc_curve(ys, scores)
eer = brentq(lambda x: 1.0 - x - interp1d(fpr, tpr)(x), 0.0, 1.0)
print(f'Test ROC-AUC : {test_auc:.4f}')
print(f'Test EER     : {eer*100:.2f}%')

# PR-AUC + F1 at best threshold
prec, rec, thr = precision_recall_curve(ys, scores)
pr_auc = sk_auc(rec, prec)
best_f1 = max(f1_score(ys, (scores>=t).astype(int)) for t in thr)
print(f'Test PR-AUC  : {pr_auc:.4f}')
print(f'Test best F1 : {best_f1:.4f}')

wandb.log({'test_auc':test_auc,'test_eer':eer,'test_pr_auc':pr_auc,'test_f1':best_f1})